# Guion de Video — Proyecto HTTP (20–25 min)

## Persona A — Primera parte completa (0:00–17:30)

### 0:00–1:30 Presentación y objetivos
- Presentación: Servidor HTTP/1.0 en Rust con enrutamiento, pools de workers (basic/cpu/io), métricas detalladas y sistema de jobs con persistencia ligera.
- Agenda: estructura de `src`, puntos clave del código, cómo correrlo y demo con `curl`.

### 1:30–4:30 Estructura del proyecto (`src`)
- `main.rs`: arranque, carga de `Config`, construcción de `Router`, `AppState::shared`, `spawn_dispatcher` y `http_listen_loop`.
- `core.rs`: núcleo HTTP (parseo del request, enrutamiento, encolado en pool y escritura de respuesta), `AppState`, serialización HTTP, métricas.
- `router.rs`: mapeo de rutas → pool y handler.
- `config.rs`: variables de entorno para puerto, workers y profundidad de colas, límites y timeouts.
- `workers.rs`: colas con backpressure y pools de workers, flags `busy` por worker.
- `metrics.rs`: métricas globales y por comando (wait/exec, avg, stddev y p50/p95/p99).
- `jobs.rs`: job store con journal JSONL, scheduler con límites por tipo y timeouts; reusa handlers.
- `handlers/`: endpoints separados en `basic.rs`, `cpu.rs`, `io.rs`, `jobs.rs`.

### 4:30–8:00 Arranque en `main.rs` y estado global
- Flujo: `Config::from_env_or_default()` → `Router::new()` → `AppState::shared(cfg, router)` → `spawn_dispatcher(shared)` → `http_listen_loop(&shared)`.
- `AppState::shared` en dos pasos: crea un `dummy` (con `Pools::new_dummy()`), luego construye `Pools::new(&dummy)` y devuelve `AppState` definitivo con pools reales.
- Beneficio: tener `Shared = Arc<AppState>` disponible al construir pools.

### 8:00–11:30 Router y manejo de conexión (visión general)
- `router.rs`: cada ruta se mapea a `Route::{Basic|Cpu|Io}` con una función handler.
- `core::handle_connection`: parsea la request-line (solo GET), arma `Request`, consulta al router y encola la tarea en el pool correspondiente. Si no existe, 404.
- `enqueue_on_pool`: captura tiempos `wait_ms` y `exec_ms`, registra métricas por comando y maneja backpressure (503 con `Retry-After`).

### 11:30–14:30 Pools y backpressure (conceptos)
- `WorkQueue`: canal MPSC + contador `pending` con `Mutex`, límite `max_depth` y `workers` hilos. Cada worker marca `busy` con `AtomicBool`.
- `submit`: si `pending >= max_depth` → error de backpressure con sugerencia de reintento.
- `Pools::new`: usa `Config` para tamaños de cola y número de workers por pool.

### 14:30–17:30 Métricas y endpoints de inspección
- Métricas globales: `accepted`, `handled`, `errors`, latencias globales y muestras.
- Métricas por comando: colecciones de muestras `wait_ms` y `exec_ms`, cálculo de avg/stddev/p50/p95/p99.
- Endpoints:
  - `/status`: snapshot de colas, workers por pool, config y timeouts.
  - `/metrics`: estadísticos por comando (latencia y colas), estimación de workers ocupados y throughput.

### 17:30–22:00 (Transición a Persona B)
- Cierre de mi parte: Arquitectura desde arriba hacia abajo lista. Ahora veremos handlers (basic/cpu/io), sistema de jobs en doble modo y demo completa.

---

## Apéndice rápido para Persona A (comandos de ejecución)

```bash
# PowerShell en Windows
cd C:\Github\ServidorHTTP\proyecto-http
$env:PORT="8080"           # opcional
$env:WORKERS_CPU="4"       # opcional
cargo run
```


## Persona B — Segunda parte completa (17:30–fin)

### 17:30–20:00 Handlers `basic.rs` (demo rápida)
- Puntos: Firma `(state, req) -> (status, content_type, body_bytes)`, validaciones y mensajes de error claros.
- Demos:
  - `GET /timestamp`
  - `GET /reverse?text=hola`
  - `GET /toupper?text=AbCd`
  - `GET /random?count=5&min=10&max=20`
  - `GET /hash?text=hola`
- Archivos: `createfile`/`deletefile` con validación de nombre (sin `..` ni separadores) y códigos 409/500 según caso.

Comandos de ejemplo:
```bash
curl http://localhost:8080/timestamp
curl "http://localhost:8080/reverse?text=hola"
curl "http://localhost:8080/toupper?text=AbCd"
curl "http://localhost:8080/random?count=5&min=10&max=20"
curl "http://localhost:8080/hash?text=hola"
```

### 20:00–24:00 Handlers `cpu.rs` y `io.rs` + doble modo job
- Doble modo en CPU/IO: `?mode=job&prio=low|normal|high` encola el trabajo y devuelve `job_id`; sin ese parámetro, ejecuta directo.
- CPU-bound ejemplos:
  - `GET /isprime?n=97&method=auto`
  - `GET /factor?n=360`
  - `GET /pi?digits=10`
  - `GET /mandelbrot?width=20&height=20&max_iter=100`
  - `GET /matrixmul?size=20&seed=123`
- IO-bound de archivos (lee desde `data/`):
  - `GET /sortfile?name=nums.txt&algo=merge|quick`
  - `GET /wordcount?name=nums.txt`
  - `GET /grep?name=nums.txt&pattern=3`
  - `GET /compress?name=nums.txt&codec=gzip|xz`
  - `GET /hashfile?name=nums.txt&algo=sha256`

Comandos de ejemplo:
```bash
curl "http://localhost:8080/isprime?n=97&method=auto"
curl "http://localhost:8080/factor?n=360"
curl "http://localhost:8080/pi?digits=10"
curl "http://localhost:8080/mandelbrot?width=20&height=20&max_iter=100"
curl "http://localhost:8080/matrixmul?size=20&seed=123"

# Crear y usar archivo
curl "http://localhost:8080/createfile?name=nums.txt&content=5,3,4,1,2&repeat=1"
curl "http://localhost:8080/sortfile?name=nums.txt&algo=merge"
curl "http://localhost:8080/wordcount?name=nums.txt"
curl "http://localhost:8080/grep?name=nums.txt&pattern=3"
curl "http://localhost:8080/compress?name=nums.txt&codec=gzip"
curl "http://localhost:8080/hashfile?name=nums.txt"
curl "http://localhost:8080/deletefile?name=nums.txt"
```

### 24:00–27:00 Sistema de Jobs (`jobs.rs`)
- Persistencia efímera: journal JSONL en `data/jobs.journal`. Recuperación al arranque y saneo de jobs `Running` → `Error("restarted")`.
- `spawn_dispatcher`: scheduler en hilo aparte:
  - Ordena pendientes por prioridad y FIFO.
  - Respeta límites de concurrencia por tipo (CPU/IO) desde `Config`.
  - Ejecuta cada job en hilo, aplica `recv_timeout` con `timeout_ms` por tarea.
- Endpoints:
  - `GET /jobs/submit?task=...&<params>&prio=...`
  - `GET /jobs/status?id=job-XXXX`
  - `GET /jobs/result?id=job-XXXX`
  - `GET /jobs/cancel?id=job-XXXX`

Comandos de ejemplo (job mode y API jobs):
```bash
# Encolar por doble modo
a=$(curl -s "http://localhost:8080/isprime?n=999983&mode=job&prio=high")
# o vía submit
a=$(curl -s "http://localhost:8080/jobs/submit?task=isprime&n=999983&prio=high")
JOB=$(echo $a | sed -E 's/.*"job_id":"([^"]+)".*/\1/')

curl "http://localhost:8080/jobs/status?id=$JOB"
curl "http://localhost:8080/jobs/result?id=$JOB"
curl "http://localhost:8080/jobs/cancel?id=$JOB"
```

### 27:00–29:00 Métricas y backpressure en vivo
- `/metrics`: p50/p95/p99, avg/stddev por comando; colas y estimación de workers ocupados; throughput.
- Backpressure: si una cola se satura (`pending >= max_depth`), respuesta 503 con `Retry-After`. 
- Mostrar cómo influye configuración:
  - Variables: `WORKERS_BASIC|CPU|IO`, `QUEUE_*`, `JOBS_QUEUE_MAX`, `TIMEOUT_CPU_MS`, `TIMEOUT_IO_MS`, `MAX_RUNNING_CPU_JOBS`, `MAX_RUNNING_IO_JOBS`.

Ejemplo rápido:
```bash
# Ajustar para forzar presión
# PowerShell (nueva instancia antes de run)
$env:QUEUE_CPU="1"; $env:WORKERS_CPU="1"
# En otra consola, lanzar múltiples /matrixmul y observar 503 y Retry-After
```

### 29:00–30:00 Cierre y próximos pasos
- Recap: arquitectura simple pero completa (HTTP 1.0, pools, métricas y jobs con persistencia).
- Posibles mejoras: soporte HTTP/1.1 keep-alive, más métodos, parsing robusto, persistencia durable, exposición Prometheus, tests de carga.

---

## Apéndice rápido para Persona B (comandos útiles)

```bash
# Endpoints de ayuda e inspección
curl http://localhost:8080/help
curl http://localhost:8080/status
curl http://localhost:8080/metrics
```
